# Exploration et preparation

Projet pedagogique, sans usage clinique ni decision sur les etudiants.


## 1. Importer les outils

Importer les outils et trouver le dossier du projet.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)


## 2. Charger et comprendre le fichier

Charger le CSV et regarder les dimensions, les premieres lignes et les types.


In [2]:
file_path = PROJECT_ROOT / "data/raw/student_mental_health.csv"
df = pd.read_csv(file_path)
print("Lignes :", df.shape[0], "- Colonnes :", df.shape[1])
display(df.head())
df.info()


Lignes : 2000 - Colonnes : 17


,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,STU0004,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,STU0005,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium


<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Student_ID                      2000 non-null   str    
 1   Age                             2000 non-null   int64  
 2   Gender                          2000 non-null   str    
 3   Year_of_Study                   2000 non-null   int64  
 4   Sleep_Hours                     1938 non-null   float64
 5   Study_Hours_Per_Week            2000 non-null   float64
 6   GPA                             1962 non-null   float64
 7   Social_Media_Hours              1942 non-null   float64
 8   Screen_Time_Hours               2000 non-null   float64
 9   Physical_Activity               2000 non-null   str    
 10  Financial_Stress                1909 non-null   float64
 11  Family_Support                  1946 non-null   float64
 12  Attendance_Rate                 1953 non-null

In [22]:
print(df.duplicated().sum())


0


## Separer le train (80 %) et le test (20 %)

Garder 80 % pour le train et 20 % pour le test. Le test reste reserve pour la fin.


In [29]:
X = df.drop(columns=["Student_ID", "Mental_Health_Risk"])
y = df["Mental_Health_Risk"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

eda = X_train.copy()
eda["Mental_Health_Risk"] = y_train
risk_order = ["Low", "Medium", "High"]
numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
category_columns = X_train.select_dtypes(exclude="number").columns.tolist()


## Statistiques descriptives et valeurs manquantes du train

Resumer les valeurs du train et compter les donnees manquantes.


In [23]:
missing = pd.DataFrame({
    "Nombre manquant": X_train.isna().sum(),
    "Pourcentage": X_train.isna().mean() * 100,
})
display(missing.round(2))


,Nombre manquant,Pourcentage
Age,0,0.00
Gender,0,0.00
Year_of_Study,0,0.00
Sleep_Hours,54,3.38
Study_Hours_Per_Week,0,0.00
GPA,36,2.25
Social_Media_Hours,48,3.00
Screen_Time_Hours,0,0.00
Physical_Activity,0,0.00
Financial_Stress,76,4.75


## Distribution de la cible et desequilibre

Les classes sont proches de 34 %, 33 %, 33 % : pas besoin de SMOTE ici.


In [27]:
class_counts = y_train.value_counts().reindex(risk_order, fill_value=0)
class_summary = pd.DataFrame({
    "Effectif": class_counts,
    "Pourcentage": class_counts / len(y_train) * 100,
})
display(class_summary.round(2))


,Effectif,Pourcentage
Mental_Health_Risk,,
Low,528,33.0
Medium,528,33.0
High,544,34.0


## Heures de la semaine

Hypothese a confirmer : sommeil, ecran et reseaux sont en h/jour. Multiplier par 7. La somme est indicative car les activites peuvent se chevaucher ; ne pas rajouter les reseaux au temps ecran.


In [32]:
eda["Sleep_Week"] = eda["Sleep_Hours"] * 7
eda["Screen_Week"] = eda["Screen_Time_Hours"] * 7
eda["Social_Week"] = eda["Social_Media_Hours"] * 7
eda["Weekly_Sum_Indicative"] = eda["Sleep_Week"] + eda["Screen_Week"] + eda["Study_Hours_Per_Week"]
weekly_columns = ["Sleep_Week", "Screen_Week", "Social_Week", "Study_Hours_Per_Week", "Weekly_Sum_Indicative"]
display(eda[weekly_columns].describe().round(2))

social_above_screen = eda["Social_Media_Hours"] > eda["Screen_Time_Hours"]
sum_above_week = eda["Weekly_Sum_Indicative"] > 168
print("Reseaux > ecran :", social_above_screen.sum(), "cas a verifier")
print("Somme indicative > 168 h :", sum_above_week.sum(), "cas a examiner")


,Sleep_Week,Screen_Week,Social_Week,Study_Hours_Per_Week,Weekly_Sum_Indicative
count,1546.00,1600.00,1552.00,1600.00,1546.00
mean,47.26,60.04,42.25,19.98,127.33
std,7.86,11.61,7.94,5.77,13.78
min,25.20,23.10,14.70,2.40,88.50
25%,42.00,51.80,37.10,16.00,117.80
50%,46.90,59.50,42.00,19.80,127.50
75%,52.50,67.90,47.60,24.00,136.67
max,70.00,98.00,70.00,38.30,170.20


Reseaux > ecran : 27 cas a verifier
Somme indicative > 168 h : 2 cas a examiner
